In [ ]:
import pandas as pd
import modules.data_prep as data_prep

## Data Prep

In [ ]:
data, data_latest_eq_map, filtered_features, gj = data_prep.data_prep_pipeline()

In [ ]:
features_list = []
for feature in filtered_features:
    props = feature['properties'].copy()
    props['geometry_type'] = feature['geometry']['type']
    props['coordinates'] = feature['geometry']['coordinates']
    features_list.append(props)

features_df = pd.DataFrame(features_list)

In [ ]:
features_df[features_df['catalog_id'].isin(['ME_TRCS372', 'EUR_TRCS372'])]

In [ ]:
me = features_df.loc[features_df['catalog_id'] == 'ME_TRCS372', 'coordinates'].values[0]
eur = features_df.loc[features_df['catalog_id'] == 'EUR_TRCS372', 'coordinates'].values[0]


def coordinates_overlap_analysis(coords1, coords2):
    """Calculate overlap statistics between two coordinate sets"""
    def flatten_coords(coords):
        if coords and isinstance(coords[0][0], list):
            return [tuple(point) for line in coords for point in line]
        else:
            return [tuple(point) for point in coords]
    
    set1 = set(flatten_coords(coords1))
    set2 = set(flatten_coords(coords2))
    
    intersection = set1.intersection(set2)
    union = set1.union(set2)
    
    overlap_count = len(intersection)
    total_coords_1 = len(set1)
    total_coords_2 = len(set2)

    jaccard_similarity = (len(intersection) / len(union) * 100) if len(union) > 0 else 0
    overlap_pct_1 = (overlap_count / total_coords_1 * 100) if total_coords_1 > 0 else 0
    overlap_pct_2 = (overlap_count / total_coords_2 * 100) if total_coords_2 > 0 else 0
    
    return {
        'has_overlap': overlap_count > 0,
        'shared_points': overlap_count,
        'total_points_fault1': total_coords_1,
        'total_points_fault2': total_coords_2,
        'jaccard_similarity_pct': round(jaccard_similarity, 2),
        'overlap_pct_fault1': round(overlap_pct_1, 2),
        'overlap_pct_fault2': round(overlap_pct_2, 2)
    }

overlap_stats = coordinates_overlap_analysis(me, eur)

print(f"Overlap Analysis:")
print(f"  Has overlap: {overlap_stats['has_overlap']}")
print(f"  Shared points: {overlap_stats['shared_points']}")
print(f"  Total points in ME_TRCS372: {overlap_stats['total_points_fault1']}")
print(f"  Total points in EUR_TRCS372: {overlap_stats['total_points_fault2']}")
print(f"  Jaccard similarity: {overlap_stats['jaccard_similarity_pct']}%")
print(f"  Overlap (% of ME_TRCS372): {overlap_stats['overlap_pct_fault1']}%")
print(f"  Overlap (% of EUR_TRCS372): {overlap_stats['overlap_pct_fault2']}%")

overlap_stats